# 00 — Granularity selection (Layer 1)

**Purpose.** Find the candle granularity Δ\* at which microstructure noise has washed out but tradable intraday horizon remains, for EUR 50Y IRS during the 08:00–17:00 CET session.

**Method.** Compute realized variance (RV) and bipower variation (BV) across the grid Δ ∈ {5, 15, 30, 60, 120} minutes, averaged over N independently simulated sessions. A signature plot of mean RV(Δ) vs Δ diagnoses microstructure noise: RV inflated at fine Δ → noise dominant; RV stabilises as Δ grows → noise regime ended. The transition identifies Δ\*.

**Output.** A recommended Δ\* for the Layer-2 mean-reversion analysis.

Constraints on Δ\*: `Δ* >= 5 min` (microstructure-noise floor, expected) and `Δ* <= 120 min` (same-day closability).

**This is a data-quality diagnostic**, not the central economic claim. If the signature plot reveals that *no* Δ in our grid gives clean data, the feasibility of the thesis itself is in question.

## Paste-ready to bQuant

Cells below take a DataFrame `df` conforming to `docs/DATA_CONTRACT.md` — 1-minute `DatetimeIndex` (CET, tz-aware), single column `50Y`, `float64` rate levels.

- **Locally:** `df` comes from `synthetic.want.ou_with_noise.simulate(...)`.
- **In bQuant:** `df` comes from a `bql` query returning the same shape. Skip the local-simulation cells (sections 1–2); paste everything from *Build candle grid* onwards.

If a cell uses a library not listed in `docs/BQUANT_COMPAT.md`, flag it before pasting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Design decisions and assumptions

Every choice below was made deliberately. If any surprises you or your supervisor, this is the first place to push back.

**Return definition.** We use **simple rate differences in basis points** — `r_Δ,t = 10^4 · (R_t − R_{t−Δ})` — not log-returns on a derived bond price. Rationale: thesis and PnL are denominated in bp, and in our regime (R ≈ 2.8%, moves of bp) the signature-plot *shape* is near-identical to the log-price-return convention up to a near-constant scaling. A reviewer preferring the academic form can swap in a 3-line change.

**Candle aggregation.** Last value in bin (`df.resample(Δ).last()`). Standard for RV / BV estimation.

**Non-overlapping returns.** Between consecutive Δ-bin closes. Overlapping returns would introduce autocorrelation that complicates inference.

**Multi-session averaging.** A single 540-min session yields only 4 non-overlapping 120-min returns — too noisy for a legible signature plot at coarse Δ. We simulate `N_SESSIONS = 100` independent sessions and plot mean RV(Δ) with ±1σ bands. *Necessity for legibility, not a rigor upgrade.*

**Stabilisation criterion for Δ\*.** Eyeball on the log-log signature plot (operator + supervisor judgment). A formal statistic (Hansen-Lunde noise-variance decomposition, Aït-Sahalia-Mykland optimal-sampling point estimator) is deferred — if the visual is ambiguous we add it.

**DGP parameters.** Literature-grounded starters (see `docs/PARAMETER_SOURCES.md`), ±2× accurate at best. If the Δ\* decision is sensitive to a factor-of-2 shift in any parameter, that fragility needs flagging before touching real bQuant data.

**Adversarial validation.** Every analysis on the `want/` DGP is mirrored on `dont_want/ou_no_noise` (section 7). A correct methodology produces an approximately flat signature plot on the null DGP. Seed of the eventual Monte Carlo type-I test (see `docs/IMPLEMENTATION_STATUS.md`).

**Not modelled** (from `docs/PARAMETER_SOURCES.md`, *Assumptions embedded in the current DGP*):
- Time-of-day volatility envelope (real vol heteroskedastic around London open, ECB hours, macro releases).
- Jumps.
- Stationary-not-drifting μ.
- OU starts at μ, not from the stationary distribution — suppresses early-session dispersion.

If any of the above turns out to matter for the Δ\* decision, we need a richer DGP before trusting the result.

## 1. Load synthetic data — LOCAL only

Skip this section when pasting into bQuant; `df` will come from a `bql` query returning a DataFrame matching `docs/DATA_CONTRACT.md`.

In [ ]:
# LOCAL ONLY — remove when pasting into bQuant.
# Parameter values are literature-grounded starters for 50Y EUR IRS intraday.
# Full derivation and citations: docs/PARAMETER_SOURCES.md
from math import log

from synthetic.want.ou_with_noise import simulate

params = {
    'mu':          0.028,        # ~2.8%; representative 50Y EUR level
    'theta':       log(2) / 60,  # half-life = 60 min (low-confidence extrapolation)
    'sigma_eff':   3.8e-5,       # stationary SD ~ 2.5 bp via sigma_eff = SD*sqrt(2 theta)
    'sigma_noise': 1.5e-5,       # noise-to-signal variance ratio ~ 0.25 at 1-min
    'n_minutes':   540,          # [08:00, 17:00) CET session
    'start':       pd.Timestamp('2024-01-02 08:00', tz='CET'),
    'seed':        0,
}

# Single-session sanity check — visually inspect that the path looks like a 50Y rate.
df_one = simulate(**params)
df_one.head()

## 2. Multi-session generation

One session gives only 4 non-overlapping 120-min returns — too noisy for a legible signature plot at coarse Δ. We generate N independent session paths and average across them.

Seeds are offset from the single-session demo above to avoid collision.

In [ ]:
N_SESSIONS = 100
SEED_OFFSET = 1  # seed 0 was used above for the single-session demo


def generate_sessions(
    n_sessions: int,
    sim_params: dict,
    sim_fn=simulate,
    seed_offset: int = 0,
) -> list:
    '''Generate N independent session paths. All parameters except seed are shared.'''
    return [sim_fn(**{**sim_params, 'seed': seed_offset + i}) for i in range(n_sessions)]


sessions = generate_sessions(N_SESSIONS, params, seed_offset=SEED_OFFSET)

first = sessions[0]
print(f'Generated {len(sessions)} sessions, {len(first)} bars each')
print(f'  [{first.index[0]} -> {first.index[-1]}]')

## 3. Build candle grid

For each session, resample to each Δ ∈ {5, 15, 30, 60, 120} min using last-value-in-bin.

Returned object: `candles[Δ]` is a list of N session DataFrames, each at granularity Δ. This is the shared input for RV / BV / signature-plot construction below.

In [ ]:
DELTAS_MIN = [5, 15, 30, 60, 120]


def resample_last(df: pd.DataFrame, minutes: int) -> pd.DataFrame:
    '''Aggregate a 1-min DataFrame to a coarser candle grid using last-value-in-bin.'''
    return df.resample(f'{minutes}min').last()


candles = {delta: [resample_last(s, delta) for s in sessions] for delta in DELTAS_MIN}

# Sanity: how many bars per session at each Δ
for delta, per_session in candles.items():
    print(f'delta={delta:>3} min  ->  {len(per_session[0])} bars per session')

## 4. Realized variance RV(Δ)

For each session at each Δ: compute non-overlapping Δ-returns in bp (`10^4 · diff`), sum of squares = `RV_session(Δ)`. Aggregate across sessions into `RV_mean(Δ)` and `RV_std(Δ)`.

**Units:** bp² summed over a 540-min session.

**Expected shape** (from `docs/PARAMETER_SOURCES.md`): under iid microstructure noise with per-return noise contribution `2 σ_noise²`, session RV at fine Δ is dominated by `n_returns · 2 σ_noise²`, which shrinks as Δ grows (fewer non-overlapping returns per session). At coarse Δ, RV converges toward the integrated variance of the efficient price.

In [ ]:
# TODO(operator-led): methodology — RV computation across candle grid and sessions.
# Expected output: pd.Series indexed by Δ with values = mean RV across sessions,
#                   and a parallel std-series for the +/- 1 sigma error bands.

## 5. Bipower variation BV(Δ)

`BV_session(Δ) = (π/2) · Σ |r_Δ,t| · |r_Δ,t−1|` over non-overlapping Δ-returns. Robust to jumps.

Our current DGP has no jumps, so `RV ≈ BV` is expected — equality is a sanity check on the code, not a discovery. A real-data run would show `RV > BV` around announcement windows.

In [ ]:
# TODO(operator-led): methodology — BV computation across candle grid and sessions.

## 6. Signature plot

Plot mean `RV(Δ)` and mean `BV(Δ)` vs `Δ` with ±1σ error bands, on log-log axes.

**Expected shape** (from `docs/PARAMETER_SOURCES.md`): declining curve at Δ < 5–7 min (microstructure noise dominates session RV), stabilising at Δ ≥ 15 min (noise regime ended). The break identifies Δ\*.

In [ ]:
# TODO(operator-led): signature-plot construction and rendering.

## 7. Adversarial sanity check on `dont_want/ou_no_noise`

Run the same RV / BV / signature-plot analysis on simulated paths from the null DGP (OU with no microstructure noise overlay). **Expected outcome: approximately flat signature plot.**

If the null signature plot is *not* flat, something in the methodology is treating OU-implied structure as if it were microstructure noise — a falsifiable bug in the RV/BV code.

Seed of the eventual Monte Carlo type-I test for `ou_no_noise` (see `docs/IMPLEMENTATION_STATUS.md`).

In [ ]:
from synthetic.dont_want.ou_no_noise import simulate as simulate_null

null_params = {k: v for k, v in params.items() if k != 'sigma_noise'}

null_sessions = generate_sessions(
    N_SESSIONS, null_params, sim_fn=simulate_null, seed_offset=10_000
)
null_candles = {
    delta: [resample_last(s, delta) for s in null_sessions] for delta in DELTAS_MIN
}

print(f'Generated {len(null_sessions)} null sessions at same N and same delta grid.')

In [ ]:
# TODO(operator-led): re-run the RV + BV + signature-plot construction on
#                     null_sessions / null_candles. Plot alongside (or in a separate
#                     subplot with shared axes) the primary signature plot.
# Expected: null curve is approximately flat.

## 8. Diagnose

Interpret the signature plot:

- Is there a Δ beyond which RV and BV stabilise? That is the noise-free regime.
- **Null curve (section 7)** — is it approximately flat, as expected for `ou_no_noise`? If not, the methodology itself is suspect and the primary signature plot cannot be trusted.
- Is there evidence of jumps (`RV >> BV`)? Not expected under our current DGP (no jumps).

In [ ]:
# TODO(operator-led): diagnostic summary.

## 9. Select Δ\*

Record the chosen Δ\* as the candle granularity used for the Layer-2 mean-reversion notebook.

**Δ\* = ?** (to be filled in)

Rationale: …

**Open concerns to carry forward:**

- **Parameter sensitivity.** Shift μ / θ / σ_eff / σ_noise each by 2× and re-run — do we pick the same Δ\*? If not, DGP-calibration uncertainty propagates into Δ\* uncertainty.
- **Null-curve flatness.** If section 7 shows a non-flat null curve, the method is detecting non-microstructure structure as if it were noise — invalidates Δ\*.
- **Heteroskedasticity.** Our DGP has stationary vol across the session; real markets do not. If Δ\* lands on the edge of the grid, confirm that a time-of-day envelope doesn't change the pick before committing.